# Bitcoin sentiment and trader performance analysis

This notebook analyzes the real Hyperliquid historical trader data and the Bitcoin fear/greed sentiment index.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path.cwd()
sentiment = pd.read_csv(ROOT / 'fear_greed_index.csv')
trader = pd.read_csv(ROOT / 'historical_data.csv')

sentiment = sentiment.rename(columns={'classification': 'Classification', 'value': 'value', 'date': 'Date'})
trader['Date'] = pd.to_datetime(trader['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce').dt.normalize()
sentiment['Date'] = pd.to_datetime(sentiment['Date'], errors='coerce').dt.normalize()

daily = (
    trader.groupby(['Date', 'Account'])
    .agg(total_closed_pnl=('Closed PnL', 'sum'), total_size=('Size Tokens', 'sum'), trade_count=('Account', 'size'))
    .reset_index()
)
merged = daily.merge(sentiment[['Date', 'Classification', 'value']], on='Date', how='left')
merged['Classification'] = merged['Classification'].fillna('Unknown')
merged.head()

In [ ]:
summary = (
    merged.groupby('Classification')
    .agg(accounts=('Account', 'nunique'), trades=('trade_count', 'sum'), avg_pnl=('total_closed_pnl', 'mean'), avg_sentiment=('value', 'mean'))
    .reset_index()
)
summary

In [ ]:
plt.style.use('seaborn-v0_8')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=summary, x='Classification', y='avg_pnl', ax=axes[0])
axes[0].set_title('Average PnL by Sentiment')
axes[0].set_ylabel('Avg closed PnL')
sns.barplot(data=summary, x='Classification', y='trades', ax=axes[1])
axes[1].set_title('Trading Activity by Sentiment')
axes[1].set_ylabel('Total trades')
plt.tight_layout()
plt.show()